# Phase 6 — Module 3 validation

Held-out datasets vs the two frozen rules. Predictions saved before training; scored after.


In [1]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
import pandas as pd
from virgo import frozen_rules as fr
from experiments import predict_module3 as pm, score_module3 as sm
fr.HELDOUT

/home/m-adam/miniconda/envs/i2v/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['citeseer_linqs',
 'proteins',
 'pubmed',
 'actor',
 'minesweeper',
 'amazon_photo']

## 1 · Predict — before training

Both frozen rules are **link-prediction** rules (Module 2 found no node-classification rule).


In [ ]:
pred, added = pm.freeze_predictions(fr.HELDOUT)
print(f"newly frozen: {added or 'none (already saved before training)'}")
display(
    pred.drop(columns=["tasks"]).round(4)   # both frozen rules are LINK-PREDICTION rules; the dataset's task list is not what is predicted
    .rename(columns={
        "homophily_adjusted": "adj_h",
        "rule1_interval": "R1 (adj_h) interval",
        "rule1_pred": "R1 (adj_h) prediction",
        "largest_component_frac": "largest_comp_frac",
        "rule2_interval": "R2 (largest_comp_frac) interval",
        "rule2_pred": "R2 (largest_comp_frac) prediction",
        "predicted_verdict": "predicted LP verdict",
    })
    .style
    .format(na_rep="—")
    .hide(axis="index")
    .set_table_styles([
        {"selector": "table", "props": [("width", "100%"), ("table-layout", "fixed"), ("font-size", "11px")]},
        {"selector": "th", "props": [("white-space", "normal"), ("word-wrap", "break-word"), ("text-align", "center"), ("padding", "4px")]},
        {"selector": "td", "props": [("white-space", "normal"), ("word-wrap", "break-word"), ("text-align", "center"), ("padding", "4px")]},
    ])
)

## 2 · Verdict vs actual — after training


In [ ]:
scored = sm.score(fr.HELDOUT)
scored.to_csv(sm.SCORED_CSV, index=False)
for r in fr.FROZEN_RULES:
    col = list(scored[f"{r.name}_correct"])
    c = [v for v in col if isinstance(v, bool)]
    skipped = sorted({str(v) for v in col if not isinstance(v, bool)})
    print(f"{r.name} ({r.predictor} {r.op} {r.point}): "
          + (f"{sum(c)}/{len(c)} correct" if c else "nothing scored yet")
          + (f"  [not scored: {', '.join(skipped)}]" if skipped else ""))
display(
    scored.round(4)
    .rename(columns={
        "homophily_adjusted": "adj_h",
        "largest_component_frac": "largest_comp_frac",
        "rule1_pred": "R1 (adj_h) prediction",
        "rule1_correct": "R1 (adj_h) correct",
        "rule2_pred": "R2 (largest_comp_frac) prediction",
        "rule2_correct": "R2 (largest_comp_frac) correct",
        "predicted_verdict": "predicted LP verdict",
        "actual_verdict": "actual LP verdict",
        "best_augmented": "best aug",
        "best_variant": "best variant",
    })
    .style
    .format(na_rep="—")
    .hide(axis="index")
    .set_table_styles([
        {"selector": "table", "props": [("width", "100%"), ("table-layout", "fixed"), ("font-size", "11px")]},
        {"selector": "th", "props": [("white-space", "normal"), ("word-wrap", "break-word"), ("text-align", "center"), ("padding", "4px")]},
        {"selector": "td", "props": [("white-space", "normal"), ("word-wrap", "break-word"), ("text-align", "center"), ("padding", "4px")]},
    ])
)